
# HuggingFace 커스텀 서비스 프레임워크 최종 노트북  
## NSMC → 카페 리뷰 운영 신호 → Store DNA → 서비스화

이 노트북의 목적은 단순히 “카페 리뷰 모델 하나”를 만드는 것이 아닙니다.

> **HuggingFace Transformers Framework 안의 Model, Tokenizer, Trainer 구조를 이해하고,  
> 어떤 도메인의 서비스 문제에도 같은 절차로 커스터마이징할 수 있는 나만의 실전 프레임워크를 만드는 것**입니다.

이번 프로젝트에서는 그 예시로 **카페 리뷰를 사장님용 운영 신호와 Store DNA 리포트로 바꾸는 서비스**를 만들었습니다.

전체 흐름은 다음과 같습니다.

```text
1. NSMC 감성분석 실습
   → HuggingFace Tokenizer / Model / Trainer 기본 구조 이해

2. 카페 리뷰 단일 라벨 분류
   → 리뷰를 긍정/부정이 아니라 운영 신호로 분류

3. 카페 리뷰 다중 라벨 분류 v1.2
   → 실제 리뷰처럼 한 문장에 여러 운영 의미가 동시에 담기는 구조 반영

4. 모델 저장
   → config.json, model.safetensors 또는 pytorch_model.bin, tokenizer 파일 생성

5. 실제 카페 리뷰 1,000개 적용
   → 학습된 모델을 서비스 추론 파이프라인으로 사용

6. Store DNA / 사장님 리포트 생성
   → 모델 출력값을 서비스 언어로 변환

7. Codex / FastAPI 서비스화
   → 노트북 코드를 API와 대시보드로 분리
```

---

## 이 노트북을 다른 서비스에 재사용하는 방법

카페 리뷰가 아니더라도 아래 8가지만 바꾸면 같은 구조를 다시 쓸 수 있습니다.

| 프레임워크 단계 | 이번 프로젝트 | 다른 서비스로 바꿀 때 |
|---|---|---|
| Problem | 카페 리뷰 운영 신호 분류 | 고객 문의 분류, 상담 품질 분석, 병원 후기 분석, 민원 분류 등 |
| Dataset | 카페 리뷰 문장 | 해당 서비스의 텍스트 데이터 |
| Label | 제품/서비스/가격/공간/재방문 신호 | 서비스 목적에 맞는 라벨 체계 |
| Tokenizer | `klue/bert-base` tokenizer | 한국어/영어/도메인에 맞는 tokenizer |
| Model | 다중 라벨 BERT 분류기 | 분류/회귀/생성/검색 모델 |
| Trainer | HuggingFace Trainer | 학습 루프 또는 PEFT/LoRA 등 |
| Metrics | micro/macro F1, precision, recall | 서비스 품질 기준에 맞는 지표 |
| Service Output | Store DNA, 사장님 액션 | 사용자가 이해하는 최종 리포트/추천/판정 |



# 0. 커스텀 프로젝트 설계 프레임워크

이 노트북의 핵심은 아래 구조를 반복해서 사용할 수 있게 만드는 것입니다.

```text
Raw Text
→ Label System
→ Dataset
→ Tokenizer
→ Model
→ Trainer
→ Evaluation
→ Save Artifacts
→ Inference Pipeline
→ Service Output
```

HuggingFace에서 특히 중요한 객체는 3개입니다.

```text
Tokenizer
- 사람의 문장을 모델이 이해할 수 있는 input_ids / attention_mask로 바꾼다.

Model
- 토큰화된 입력을 받아 라벨별 점수(logits)를 만든다.

Trainer
- 데이터셋, 모델, 평가 지표, 학습 설정을 묶어서 fine-tuning을 수행한다.
```

서비스화를 위해서는 여기에 3개가 더 필요합니다.

```text
Label Map
- 모델의 숫자 출력이 어떤 서비스 의미인지 연결한다.

Threshold
- 다중 라벨에서 어떤 확률 이상을 라벨로 볼지 결정한다.

Report Layer
- 모델 출력을 사용자가 이해하는 문장과 액션으로 바꾼다.
```



# 1. 프로젝트 발전 흐름

## 1-1. NSMC 감성분석에서 얻은 것

처음에는 NSMC 영화 리뷰 감성분석으로 출발했습니다.  
이 단계의 목적은 모델 성능 자체보다 HuggingFace의 기본 작업 순서를 익히는 것이었습니다.

```text
문장
→ Tokenizer
→ Dataset
→ Model
→ Trainer
→ Evaluation
```

NSMC에서는 긍정/부정이라는 단순한 라벨을 사용했습니다.

## 1-2. 카페 리뷰 단일 라벨 분류

그 다음에는 카페 리뷰를 긍정/부정이 아니라 운영 신호로 바꾸었습니다.

```text
"커피는 맛있는데 가격이 조금 비싸요."
```

이 문장을 단순히 부정으로 보지 않고,

```text
제품/메뉴 강점 + 가격 저항
```

으로 해석하는 방향입니다.

## 1-3. 다중 라벨 분류 v1.2

실제 리뷰는 하나의 의미만 담지 않습니다.  
그래서 단일 라벨 분류에서 다중 라벨 분류로 확장했습니다.

예시:

```text
"디저트는 맛있고 분위기도 좋은데 웨이팅이 길었어요."
→ 제품/메뉴 강점
→ 공간/분위기 강점
→ 대기/혼잡 리스크
```

이 구조가 실제 서비스에 더 적합합니다.



# 2. 실행 방법

이 노트북은 두 가지 모드로 사용할 수 있습니다.

## A. 모델 학습까지 다시 실행하는 모드

아래 학습 셀들을 순서대로 실행하면 모델이 저장됩니다.

생성되는 핵심 파일:

```text
models/cafe_review_signal_multilabel_v1_2/
├─ config.json
├─ model.safetensors 또는 pytorch_model.bin
├─ tokenizer.json
├─ tokenizer_config.json
├─ special_tokens_map.json
├─ metadata.json
├─ operation_labels.json
├─ label_thresholds.json
├─ label2id.json
└─ id2label.json
```

## B. 이미 저장된 모델로 실제 리뷰만 분석하는 모드

학습을 다시 하지 않고, 아래 후반부의 **저장 모델 로드 → 실제 리뷰 1,000개 분석 → Store DNA 생성** 부분만 실행하면 됩니다.

서비스화 단계에서는 B 모드가 중요합니다.


# 3. 카페 리뷰 운영 신호 다중 라벨 모델 학습 파트

아래 섹션은 기존 `cafe_review_v1_2_FINAL_model_save_ready` 노트북의 핵심 학습 파이프라인입니다.

목표는 다중 라벨 모델을 학습하고, 서비스에서 로드할 수 있는 HuggingFace 모델 산출물을 저장하는 것입니다.

## 0. 실행 순서

1. 패키지 설치 셀 실행
2. **Kernel Restart / 런타임 다시 시작**
3. 노트북을 처음부터 다시 실행
4. 학습 완료 후 threshold 튜닝 결과 확인
5. best threshold / label-wise threshold / top-k 결과 비교
6. 샘플 리뷰 예측과 Store DNA 점수 확인

이 노트북은 checkpoint 저장 오류를 피하기 위해 `save_strategy="no"`를 사용합니다.

## 1. 안정 패키지 설치 셀

RunPod RTX 4090 / CUDA 12.4 기준 안정 조합입니다. 설치 후 반드시 커널을 재시작하세요.

In [ ]:
# RunPod RTX 4090 / CUDA 12.4 기준 안정 설치 셀
# 설치 후 반드시 Kernel Restart / 런타임 다시 시작을 하세요.

!pip uninstall -y transformers torch torchvision torchaudio accelerate datasets evaluate -q
!pip install -q torch==2.4.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.33.0 evaluate scikit-learn pandas numpy matplotlib

## 2. 라이브러리 불러오기와 실행 환경 확인

이 셀은 실제 GPU와 패키지 버전을 확인합니다.

In [ ]:
import os, re, json, time, random, shutil, ast
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import transformers
import datasets
import accelerate

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs" / "cafe_review_signal_multilabel_v1_2"
MODEL_DIR = BASE_DIR / "models" / "cafe_review_signal_multilabel_v1_2"
SERVICE_DIR = BASE_DIR / "app"
for p in [DATA_DIR, OUTPUT_DIR, MODEL_DIR, SERVICE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. 디스크 용량 확인과 checkpoint 정리

이전 실험에서 생긴 checkpoint와 임시 파일이 디스크를 차지할 수 있으므로 정리합니다.

In [ ]:
import os
from pathlib import Path

PROJECT_NAME = "cafe_review_signal_multilabel_v1_2"
PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / PROJECT_NAME
MODEL_DIR = PROJECT_ROOT / "models" / PROJECT_NAME

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("프로젝트 기본 경로 설정 완료")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"MODEL_DIR: {MODEL_DIR}")

print("\n현재 작업 폴더 파일 목록:")
for item in sorted(PROJECT_ROOT.iterdir()):
    print("-", item.name)

## 4. 운영 신호 라벨 정의

서비스에서 사용할 운영 신호 라벨입니다. 이번 노트북은 이 라벨들을 **다중 라벨**로 예측합니다.

In [ ]:
OPERATION_LABELS = [
    "product_strength",
    "product_risk",
    "service_strength",
    "service_risk",
    "price_resistance",
    "price_value_strength",
    "waiting_risk",
    "space_strength",
    "space_risk",
    "cleanliness_strength",
    "cleanliness_risk",
    "revisit_intent",
    "revisit_risk",
    "operation_efficiency",
]

LABEL_KO = {
    "product_strength": "제품 경쟁력",
    "product_risk": "제품 품질 리스크",
    "service_strength": "서비스 강점",
    "service_risk": "서비스 리스크",
    "price_resistance": "가격 저항",
    "price_value_strength": "가격 대비 만족",
    "waiting_risk": "대기/혼잡 리스크",
    "space_strength": "공간 강점",
    "space_risk": "공간 리스크",
    "cleanliness_strength": "청결 강점",
    "cleanliness_risk": "청결 리스크",
    "revisit_intent": "재방문 의도",
    "revisit_risk": "재방문 리스크",
    "operation_efficiency": "운영 효율 신호",
}

pd.DataFrame({"label": OPERATION_LABELS, "korean": [LABEL_KO[x] for x in OPERATION_LABELS]})

## 5. 데이터 로드/생성 함수

우선순위는 다음입니다.

1. `data/cafe_review_signal_dataset_v2_multilabel.csv`
2. `data/cafe_review_signal_dataset_v1.csv`
3. 둘 다 없으면 실험용 데이터를 자동 생성

v1.2는 복합 라벨 문장을 더 많이 포함하도록 데이터를 보강합니다.

In [ ]:
def normalize_signal(value):
    """operation_signals 값을 항상 list[str] 형태로 안전하게 변환합니다."""
    if isinstance(value, (list, tuple, set, np.ndarray)):
        items = []
        for x in list(value):
            items.extend(normalize_signal(x))
        return sorted(set(items))

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    if isinstance(value, str):
        text = value.strip()
        if not text or text.lower() in {"nan", "none", "null"}:
            return []
        # 리스트 문자열 처리: "['a', 'b']"
        if text.startswith("[") and text.endswith("]"):
            try:
                parsed = ast.literal_eval(text)
                return normalize_signal(parsed)
            except Exception:
                pass
        # 구분자 처리
        for sep in [";", ",", "|", "/"]:
            text = text.replace(sep, ";")
        return sorted(set([x.strip() for x in text.split(";") if x.strip()]))

    return []


def infer_signals_from_text(text):
    text = str(text)
    signals = []
    rules = [
        ("product_strength", ["맛있", "고소", "진하", "디저트가 좋", "음료가 좋", "메뉴가 좋", "퀄리티", "풍미"]),
        ("product_risk", ["맛없", "싱겁", "탄맛", "별로", "품질", "차갑", "묽", "누락"]),
        ("service_strength", ["친절", "상냥", "응대가 좋", "설명", "배려"]),
        ("service_risk", ["불친절", "응대가 아쉬", "직원이", "무뚝뚝", "불쾌", "대답"]),
        ("price_resistance", ["비싸", "가격이", "가성비가 아쉽", "부담", "양이 적"]),
        ("price_value_strength", ["가성비", "가격 대비", "합리", "구성이 좋"]),
        ("waiting_risk", ["기다", "대기", "오래 걸", "줄이", "혼잡", "붐벼"]),
        ("space_strength", ["분위기", "인테리어", "좌석이 편", "머물기", "공간이 좋", "뷰가"]),
        ("space_risk", ["좁", "시끄", "자리 없", "불편", "답답"]),
        ("cleanliness_strength", ["깔끔", "깨끗", "청결"]),
        ("cleanliness_risk", ["지저분", "테이블 정리", "더럽", "위생", "냄새"]),
        ("revisit_intent", ["다시", "또 오", "재방문", "추천", "방문하고 싶"]),
        ("revisit_risk", ["다시는", "안 갈", "한 번이면", "재방문은", "추천하기 어렵"]),
        ("operation_efficiency", ["빠르게", "회전", "동선", "정리", "처리", "효율"]),
    ]
    for label, kws in rules:
        if any(kw in text for kw in kws):
            signals.append(label)
    return sorted(set(signals))


def base_single_label_templates():
    templates = {
        "product_strength": ["커피 향이 진하고 맛이 좋아요.", "디저트가 신선하고 만족스러웠어요.", "음료 퀄리티가 기대 이상이었어요."],
        "product_risk": ["커피가 너무 묽고 맛이 아쉬웠어요.", "디저트가 오래된 느낌이라 실망했어요.", "음료 맛이 일정하지 않았어요."],
        "service_strength": ["직원분이 친절하게 응대해 주셨어요.", "주문 설명이 자세해서 좋았어요.", "바쁜 시간에도 직원이 상냥했어요."],
        "service_risk": ["직원 응대가 무뚝뚝해서 아쉬웠어요.", "문의했는데 대답이 불친절했어요.", "서비스가 기대보다 부족했어요."],
        "price_resistance": ["가격이 조금 비싸게 느껴졌어요.", "양에 비해 가격 부담이 있었어요.", "가격 대비 만족도가 낮았어요."],
        "price_value_strength": ["가격 대비 구성이 좋아 만족했어요.", "가성비가 좋아 자주 오고 싶어요.", "이 정도 가격이면 충분히 괜찮아요."],
        "waiting_risk": ["주말에는 대기 시간이 너무 길었어요.", "주문 후 음료가 나오기까지 오래 걸렸어요.", "사람이 많아 매장이 혼잡했어요."],
        "space_strength": ["분위기가 좋아 오래 머물기 좋은 카페였어요.", "좌석이 편하고 공간이 아늑했어요.", "인테리어가 예뻐서 사진 찍기 좋았어요."],
        "space_risk": ["좌석 간격이 좁아서 불편했어요.", "매장이 너무 시끄러워 대화하기 어려웠어요.", "자리가 부족해서 오래 머물기 힘들었어요."],
        "cleanliness_strength": ["매장이 깔끔하고 테이블도 깨끗했어요.", "화장실까지 청결해서 좋았어요.", "전체적으로 관리가 잘 된 느낌이었어요."],
        "cleanliness_risk": ["테이블 정리가 늦어서 지저분했어요.", "매장 바닥과 좌석이 깨끗하지 않았어요.", "위생 관리가 조금 아쉬웠어요."],
        "revisit_intent": ["다음에도 다시 방문하고 싶어요.", "친구에게 추천하고 싶은 카페예요.", "근처에 오면 또 들를 것 같아요."],
        "revisit_risk": ["다시는 방문하지 않을 것 같아요.", "한 번 방문으로 충분하다고 느꼈어요.", "재방문 의사는 별로 없어요."],
        "operation_efficiency": ["주문 처리와 픽업 동선이 효율적이었어요.", "바쁜 시간에도 음료가 빠르게 나왔어요.", "직원들의 정리와 운영이 매끄러웠어요."],
    }
    rows = []
    for label, texts in templates.items():
        for i, text in enumerate(texts):
            rows.append({"id": f"base_{label}_{i}", "text": text, "operation_signals": [label]})
    return rows


def composite_templates():
    combos = [
        (["product_strength", "service_risk"], ["커피는 맛있는데 직원 응대가 조금 아쉬웠어요.", "디저트는 좋았지만 직원이 무뚝뚝했어요."]),
        (["price_resistance", "product_strength", "revisit_intent"], ["가격은 조금 비싸지만 디저트가 맛있어서 또 오고 싶어요.", "음료 가격은 부담되지만 맛이 좋아 다시 방문할 것 같아요."]),
        (["waiting_risk", "service_strength"], ["주말에는 오래 기다렸지만 직원분은 친절했어요.", "대기 시간은 길었지만 안내가 친절해서 괜찮았어요."]),
        (["space_strength", "revisit_intent"], ["분위기가 좋아서 오래 머물고 다시 오고 싶어요.", "공간이 아늑해서 다음에도 방문하고 싶어요."]),
        (["product_strength", "cleanliness_risk"], ["커피는 맛있지만 테이블이 지저분해서 아쉬웠어요.", "디저트는 좋았는데 매장 청결은 아쉬웠어요."]),
        (["service_risk", "waiting_risk"], ["직원 응대가 아쉽고 주문도 오래 기다렸어요.", "대기 시간이 길고 서비스도 조금 불친절했어요."]),
        (["price_value_strength", "space_strength", "revisit_intent"], ["가성비도 좋고 분위기도 좋아서 또 오고 싶어요.", "가격 대비 만족스럽고 공간이 좋아 재방문하고 싶어요."]),
        (["space_risk", "product_strength"], ["커피 맛은 좋았지만 좌석이 좁아서 불편했어요.", "음료는 맛있는데 매장이 너무 시끄러웠어요."]),
        (["operation_efficiency", "service_strength"], ["음료가 빠르게 나오고 직원 안내도 친절했어요.", "주문 동선이 편하고 직원 응대도 좋았어요."]),
        (["product_risk", "revisit_risk"], ["음료 맛이 아쉬워서 다시 방문하진 않을 것 같아요.", "디저트 품질이 실망스러워 재방문 의사는 없어요."]),
        (["cleanliness_strength", "space_strength"], ["매장이 깔끔하고 분위기도 좋아서 편하게 머물렀어요.", "공간이 예쁘고 테이블도 깨끗했어요."]),
        (["price_resistance", "space_strength"], ["가격은 비싼 편이지만 분위기는 정말 좋았어요.", "음료 가격은 부담되지만 공간은 만족스러웠어요."]),
    ]
    rows = []
    idx = 0
    for labels, texts in combos:
        for text in texts:
            rows.append({"id": f"composite_{idx:04d}", "text": text, "operation_signals": labels})
            idx += 1
    return rows


def build_synthetic_dataset(repeat=8):
    rows = []
    seed_rows = base_single_label_templates() + composite_templates()
    modifiers = ["", " 전반적으로", " 오늘 방문했는데", " 개인적으로", " 친구와 함께 갔을 때"]
    tails = ["", "라고 느꼈어요.", "라는 생각이 들었습니다.", "점이 기억에 남아요.", "부분이 인상적이었어요."]
    idx = 0
    for r in seed_rows:
        text0 = r["text"].rstrip(".")
        for k in range(repeat):
            prefix = modifiers[k % len(modifiers)]
            tail = tails[k % len(tails)]
            text = f"{prefix} {text0} {tail}".strip()
            rows.append({"id": f"syn_{idx:05d}", "text": re.sub(r"\s+", " ", text), "operation_signals": list(r["operation_signals"])})
            idx += 1
    return pd.DataFrame(rows)


def load_or_create_dataset():
    p2 = DATA_DIR / "cafe_review_signal_dataset_v2_multilabel.csv"
    p1 = DATA_DIR / "cafe_review_signal_dataset_v1.csv"
    if p2.exists():
        print("v2 데이터셋 로드:", p2)
        df = pd.read_csv(p2)
    elif p1.exists():
        print("v1 데이터셋 로드 후 다중 라벨 변환:", p1)
        df = pd.read_csv(p1)
    else:
        print("데이터셋이 없어 실험용 데이터를 생성합니다.")
        df = build_synthetic_dataset(repeat=8)
        df.to_csv(p2, index=False, encoding="utf-8-sig")
        print("생성 저장:", p2)

    if "operation_signals" in df.columns:
        df["operation_signals"] = df["operation_signals"].apply(normalize_signal)
    elif "operation_signal" in df.columns:
        df["operation_signals"] = df["operation_signal"].apply(normalize_signal)
    else:
        df["operation_signals"] = df["text"].apply(infer_signals_from_text)

    df["operation_signals"] = df["operation_signals"].apply(lambda xs: sorted(set([x for x in xs if x in OPERATION_LABELS])))
    df = df[df["operation_signals"].apply(len) > 0].copy()
    df["text"] = df["text"].astype(str).str.strip()
    df = df[df["text"].str.len() > 0].copy()
    return df.reset_index(drop=True)

df = load_or_create_dataset()
print("초기 데이터 수:", len(df))
df.head()

## 6. 중복 제거와 복합 라벨 데이터 보강

v1.1에서 중복이 많이 확인되었으므로 text 기준으로 중복을 제거합니다. 이후 부족한 라벨과 복합 라벨 문장을 보강합니다.

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
after = len(df)
print(f"중복 제거 전: {before}")
print(f"중복 제거 후: {after}")
print(f"제거된 중복 수: {before - after}")

# 복합 라벨 샘플 추가
composite_df = pd.DataFrame(composite_templates())
# 다양한 표현으로 반복 생성
extra_composites = []
for rep in range(10):
    for _, r in composite_df.iterrows():
        text = r["text"].rstrip(".")
        variants = [
            f"오늘 방문했는데 {text}.",
            f"개인적으로 {text}라고 느꼈어요.",
            f"친구와 함께 갔을 때 {text}.",
            f"전체적으로 {text} 부분이 기억에 남아요.",
        ]
        extra_composites.append({"id": f"extra_comp_{rep}_{_}", "text": variants[rep % len(variants)], "operation_signals": r["operation_signals"]})
extra_df = pd.DataFrame(extra_composites)

# 라벨별 최소 샘플 수 보강
MIN_PER_LABEL = 90
label_counter = Counter()
for labels in df["operation_signals"]:
    label_counter.update(labels)

synthetic_pool = build_synthetic_dataset(repeat=12)
augment_rows = []
for label in OPERATION_LABELS:
    need = max(0, MIN_PER_LABEL - label_counter.get(label, 0))
    if need > 0:
        candidates = synthetic_pool[synthetic_pool["operation_signals"].apply(lambda xs: label in xs)].head(need)
        augment_rows.append(candidates)
        print(f"보강: {label} {need}개")

parts = [df, extra_df]
if augment_rows:
    parts.extend(augment_rows)

df = pd.concat(parts, ignore_index=True)
df["operation_signals"] = df["operation_signals"].apply(normalize_signal)
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

# 라벨이 없는 행 제거
df["operation_signals"] = df["operation_signals"].apply(lambda xs: sorted(set([x for x in xs if x in OPERATION_LABELS])))
df = df[df["operation_signals"].apply(len) > 0].reset_index(drop=True)

# 저장
v12_data_path = DATA_DIR / "cafe_review_signal_dataset_v12_multilabel_augmented.csv"
df.to_csv(v12_data_path, index=False, encoding="utf-8-sig")

label_counter = Counter()
for labels in df["operation_signals"]:
    label_counter.update(labels)

avg_labels = np.mean([len(x) for x in df["operation_signals"]])
print("최종 데이터 수:", len(df))
print("샘플당 평균 라벨 수:", round(avg_labels, 3))
print("저장:", v12_data_path)

pd.DataFrame(label_counter.items(), columns=["label", "count"]).sort_values("count", ascending=False)

## 7. 다중 라벨 벡터 변환과 분리

다중 라벨 분류에서는 각 라벨을 0/1 벡터로 변환합니다.

In [ ]:
mlb = MultiLabelBinarizer(classes=OPERATION_LABELS)
Y = mlb.fit_transform(df["operation_signals"]).astype(np.float32)
label_cols = [f"label_{label}" for label in OPERATION_LABELS]
for i, label in enumerate(OPERATION_LABELS):
    df[f"label_{label}"] = Y[:, i]

print("라벨 벡터 shape:", Y.shape)
print("샘플당 평균 positive label 수:", Y.sum(axis=1).mean())
print("총 positive label 수:", int(Y.sum()))

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, shuffle=True)
valid_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, shuffle=True)

print("train:", len(train_df))
print("valid:", len(valid_df))
print("test:", len(test_df))

def label_summary(split_df, name):
    arr = split_df[label_cols].values
    return pd.DataFrame({
        "split": name,
        "label": OPERATION_LABELS,
        "positive_count": arr.sum(axis=0).astype(int),
    })

split_summary = pd.concat([
    label_summary(train_df, "train"),
    label_summary(valid_df, "valid"),
    label_summary(test_df, "test"),
], ignore_index=True)

split_summary.to_csv(OUTPUT_DIR / "label_distribution_by_split.csv", index=False, encoding="utf-8-sig")
split_summary.pivot(index="label", columns="split", values="positive_count")

## 8. HuggingFace Dataset 구성과 Tokenizer 적용

기존 NSMC 실험과 같은 `klue/bert-base`를 사용하되, 문제 유형은 `multi_label_classification`입니다.

In [ ]:
MODEL_NAME = "klue/bert-base"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_hf_dataset(split_df):
    tmp = split_df[["text"] + label_cols].copy()
    tmp["labels"] = tmp[label_cols].values.tolist()
    tmp = tmp[["text", "labels"]]
    return Dataset.from_pandas(tmp, preserve_index=False)

raw_datasets = DatasetDict({
    "train": make_hf_dataset(train_df),
    "valid": make_hf_dataset(valid_df),
    "test": make_hf_dataset(test_df),
})

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized_datasets = raw_datasets.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

## 9. Weighted Loss 적용 모델과 Trainer

v1.1에서는 모델이 모든 라벨을 낮은 확률로 보는 문제가 있었습니다. 이번에는 `pos_weight`를 사용해 positive label의 학습 비중을 높입니다.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(OPERATION_LABELS),
    problem_type="multi_label_classification",
    id2label={i: label for i, label in enumerate(OPERATION_LABELS)},
    label2id={label: i for i, label in enumerate(OPERATION_LABELS)},
)

train_labels = train_df[label_cols].values.astype(np.float32)
pos_counts = train_labels.sum(axis=0)
neg_counts = len(train_labels) - pos_counts
# 너무 큰 가중치는 학습을 불안정하게 만들 수 있어 상한을 둡니다.
pos_weight = np.clip(neg_counts / np.maximum(pos_counts, 1), 1.0, 8.0).astype(np.float32)
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32)

pos_weight_df = pd.DataFrame({
    "label": OPERATION_LABELS,
    "positive_count": pos_counts.astype(int),
    "negative_count": neg_counts.astype(int),
    "pos_weight": pos_weight,
})
pos_weight_df.to_csv(OUTPUT_DIR / "pos_weight_by_label.csv", index=False, encoding="utf-8-sig")
pos_weight_df

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

DEFAULT_THRESHOLD = 0.30

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid(logits)
    preds = (probs >= DEFAULT_THRESHOLD).astype(int)
    labels = labels.astype(int)
    empty_rate = (preds.sum(axis=1) == 0).mean()
    return {
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "micro_precision": precision_score(labels, preds, average="micro", zero_division=0),
        "micro_recall": recall_score(labels, preds, average="micro", zero_division=0),
        "empty_prediction_rate": empty_rate,
    }

class WeightedMultilabelTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(logits.device))
        loss = loss_fct(logits, labels.float())
        return (loss, outputs) if return_outputs else loss

## 10. 학습 실행

중간 checkpoint는 저장하지 않습니다. 최종 모델만 별도 저장합니다.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "trainer_outputs"),
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = WeightedMultilabelTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["valid"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    pos_weight=pos_weight_tensor,
)

start_time = time.time()
train_result = trainer.train()
training_time = time.time() - start_time
print(f"학습 시간: {training_time:.2f}초")

## 11. 기본 Test 평가

기본 threshold는 0.30으로 둡니다. 하지만 최종 판단은 다음 셀의 threshold 튜닝 결과를 기준으로 합니다.

In [ ]:
test_metrics_default = trainer.evaluate(tokenized_datasets["test"])
test_metrics_default["training_time_seconds"] = training_time
with open(OUTPUT_DIR / "test_metrics_default_threshold_030.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics_default, f, ensure_ascii=False, indent=2)
test_metrics_default

## 12. Threshold 튜닝

v1.1의 핵심 문제는 0.5 threshold가 너무 높아 빈 예측이 생긴 것입니다. 여기서는 valid set에서 여러 threshold를 테스트하고 최적값을 찾습니다.

In [ ]:
def metrics_at_threshold(y_true, probs, threshold):
    preds = (probs >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "micro_f1": f1_score(y_true, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, preds, average="macro", zero_division=0),
        "micro_precision": precision_score(y_true, preds, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, preds, average="micro", zero_division=0),
        "empty_prediction_rate": float((preds.sum(axis=1) == 0).mean()),
        "avg_predicted_labels": float(preds.sum(axis=1).mean()),
    }

valid_pred = trainer.predict(tokenized_datasets["valid"])
valid_probs = sigmoid(valid_pred.predictions)
valid_labels = np.array(valid_df[label_cols].values).astype(int)

test_pred = trainer.predict(tokenized_datasets["test"])
test_probs = sigmoid(test_pred.predictions)
test_labels = np.array(test_df[label_cols].values).astype(int)

thresholds = np.round(np.arange(0.10, 0.55, 0.05), 2)
valid_threshold_results = pd.DataFrame([metrics_at_threshold(valid_labels, valid_probs, t) for t in thresholds])
valid_threshold_results = valid_threshold_results.sort_values(["micro_f1", "macro_f1"], ascending=False).reset_index(drop=True)

best_threshold = float(valid_threshold_results.iloc[0]["threshold"])
print("best_threshold:", best_threshold)
valid_threshold_results.to_csv(OUTPUT_DIR / "valid_threshold_tuning_results.csv", index=False, encoding="utf-8-sig")
valid_threshold_results

In [ ]:
test_threshold_results = pd.DataFrame([metrics_at_threshold(test_labels, test_probs, t) for t in thresholds])
test_threshold_results.to_csv(OUTPUT_DIR / "test_threshold_tuning_results.csv", index=False, encoding="utf-8-sig")

test_best_metrics = metrics_at_threshold(test_labels, test_probs, best_threshold)
with open(OUTPUT_DIR / "test_metrics_best_global_threshold.json", "w", encoding="utf-8") as f:
    json.dump(test_best_metrics, f, ensure_ascii=False, indent=2)

test_threshold_results

## 13. 라벨별 Threshold 탐색

운영 신호마다 적절한 threshold가 다를 수 있습니다. 예를 들어 `waiting_risk`와 `revisit_intent`의 확률 분포는 다를 수 있습니다.

In [ ]:
def best_threshold_for_each_label(y_true, probs, thresholds):
    rows = []
    label_thresholds = {}
    for j, label in enumerate(OPERATION_LABELS):
        best = {"label": label, "threshold": 0.30, "f1": -1, "precision": 0, "recall": 0, "support": int(y_true[:, j].sum())}
        for t in thresholds:
            pred = (probs[:, j] >= t).astype(int)
            f1 = f1_score(y_true[:, j], pred, zero_division=0)
            precision = precision_score(y_true[:, j], pred, zero_division=0)
            recall = recall_score(y_true[:, j], pred, zero_division=0)
            if f1 > best["f1"]:
                best.update({"threshold": float(t), "f1": float(f1), "precision": float(precision), "recall": float(recall)})
        rows.append(best)
        label_thresholds[label] = best["threshold"]
    return pd.DataFrame(rows), label_thresholds

label_threshold_df, label_thresholds = best_threshold_for_each_label(valid_labels, valid_probs, thresholds)
label_threshold_df.to_csv(OUTPUT_DIR / "label_wise_thresholds_valid.csv", index=False, encoding="utf-8-sig")
with open(OUTPUT_DIR / "label_wise_thresholds.json", "w", encoding="utf-8") as f:
    json.dump(label_thresholds, f, ensure_ascii=False, indent=2)
label_threshold_df

In [ ]:
def predict_with_label_thresholds(probs, label_thresholds):
    preds = np.zeros_like(probs, dtype=int)
    for j, label in enumerate(OPERATION_LABELS):
        preds[:, j] = (probs[:, j] >= label_thresholds[label]).astype(int)
    return preds

labelwise_preds = predict_with_label_thresholds(test_probs, label_thresholds)
labelwise_metrics = {
    "micro_f1": f1_score(test_labels, labelwise_preds, average="micro", zero_division=0),
    "macro_f1": f1_score(test_labels, labelwise_preds, average="macro", zero_division=0),
    "micro_precision": precision_score(test_labels, labelwise_preds, average="micro", zero_division=0),
    "micro_recall": recall_score(test_labels, labelwise_preds, average="micro", zero_division=0),
    "empty_prediction_rate": float((labelwise_preds.sum(axis=1) == 0).mean()),
    "avg_predicted_labels": float(labelwise_preds.sum(axis=1).mean()),
}
with open(OUTPUT_DIR / "test_metrics_labelwise_threshold.json", "w", encoding="utf-8") as f:
    json.dump(labelwise_metrics, f, ensure_ascii=False, indent=2)
labelwise_metrics

## 14. Top-k 평가

threshold가 낮게 잡히거나 높게 잡히면 서비스 해석이 흔들릴 수 있습니다. 그래서 정답 라벨이 top-k 후보 안에 들어가는지도 확인합니다.

In [ ]:
def topk_metrics(y_true, probs, ks=(1,2,3,5)):
    rows = []
    top_order = np.argsort(-probs, axis=1)
    true_sets = [set(np.where(row == 1)[0]) for row in y_true]
    for k in ks:
        hit_any = []
        cover_all = []
        for i in range(len(y_true)):
            pred_set = set(top_order[i, :k])
            true_set = true_sets[i]
            hit_any.append(len(pred_set & true_set) > 0 if true_set else False)
            cover_all.append(true_set.issubset(pred_set) if true_set else False)
        rows.append({
            "top_k": k,
            "hit_any_rate": float(np.mean(hit_any)),
            "cover_all_true_labels_rate": float(np.mean(cover_all)),
        })
    return pd.DataFrame(rows)

topk_df = topk_metrics(test_labels, test_probs, ks=(1,2,3,5))
topk_df.to_csv(OUTPUT_DIR / "test_topk_metrics.csv", index=False, encoding="utf-8-sig")
topk_df

## 15. 라벨별 성능 리포트

best global threshold와 label-wise threshold 기준의 라벨별 성능을 각각 저장합니다.

In [ ]:
best_global_preds = (test_probs >= best_threshold).astype(int)

report_global = classification_report(
    test_labels,
    best_global_preds,
    target_names=OPERATION_LABELS,
    zero_division=0,
    output_dict=True,
)
report_labelwise = classification_report(
    test_labels,
    labelwise_preds,
    target_names=OPERATION_LABELS,
    zero_division=0,
    output_dict=True,
)

report_global_df = pd.DataFrame(report_global).T
report_labelwise_df = pd.DataFrame(report_labelwise).T
report_global_df.to_csv(OUTPUT_DIR / "classification_report_best_global_threshold.csv", encoding="utf-8-sig")
report_labelwise_df.to_csv(OUTPUT_DIR / "classification_report_labelwise_threshold.csv", encoding="utf-8-sig")

report_labelwise_df

## 16. 오답/미탐지 분석

빈 예측, 놓친 라벨, 과잉 예측을 확인합니다.

In [ ]:
def row_label_list(binary_row):
    return [OPERATION_LABELS[j] for j, v in enumerate(binary_row) if v == 1]

analysis_rows = []
test_texts = test_df["text"].tolist()
true_label_lists = test_df["operation_signals"].tolist()

for i, text in enumerate(test_texts):
    true_set = set(true_label_lists[i])
    pred_set = set(row_label_list(labelwise_preds[i]))
    top3_idx = np.argsort(-test_probs[i])[:3]
    top3 = [(OPERATION_LABELS[j], float(test_probs[i, j])) for j in top3_idx]
    analysis_rows.append({
        "text": text,
        "true_labels": ";".join(sorted(true_set)),
        "pred_labels_labelwise_threshold": ";".join(sorted(pred_set)),
        "missed": ";".join(sorted(true_set - pred_set)),
        "extra": ";".join(sorted(pred_set - true_set)),
        "top3": json.dumps(top3, ensure_ascii=False),
        "is_empty_pred": len(pred_set) == 0,
    })

error_df = pd.DataFrame(analysis_rows)
error_df.to_csv(OUTPUT_DIR / "test_prediction_analysis_labelwise_threshold.csv", index=False, encoding="utf-8-sig")
print("빈 예측 비율:", error_df["is_empty_pred"].mean())
error_df.head(20)

## 17. 문장/절 단위 분리와 서비스형 예측 함수

복합 리뷰를 절 단위로 나눈 뒤 각 절의 운영 신호를 예측합니다. 서비스에서는 이 방식이 단일 문장 예측보다 자연스럽습니다.

In [ ]:
def split_review_clauses(text):
    text = str(text).strip()
    patterns = [
        r"는데\s*", r"지만\s*", r"다만\s*", r"하지만\s*", r"그런데\s*",
        r"그리고\s*", r"또\s*", r"반면\s*", r",", r"\."
    ]
    regex = "|".join(patterns)
    parts = re.split(regex, text)
    parts = [p.strip() for p in parts if len(p.strip()) >= 3]
    return parts if parts else [text]


def predict_multilabel_texts(texts, mode="labelwise", threshold=None, top_k=3, fallback_top_k=2):
    if isinstance(texts, str):
        texts = [texts]

    enc = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
        return_tensors="pt",
    )
    device = model.device
    enc = {k: v.to(device) for k, v in enc.items()}
    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits.detach().cpu().numpy()
    probs = sigmoid(logits)

    results = []
    for text, prob in zip(texts, probs):
        if mode == "labelwise":
            selected = [label for j, label in enumerate(OPERATION_LABELS) if prob[j] >= label_thresholds[label]]
        else:
            th = best_threshold if threshold is None else threshold
            selected = [label for j, label in enumerate(OPERATION_LABELS) if prob[j] >= th]

        top_idx = np.argsort(-prob)[:top_k]
        top_labels = [{"label": OPERATION_LABELS[j], "label_ko": LABEL_KO[OPERATION_LABELS[j]], "score": float(prob[j])} for j in top_idx]

        # 서비스에서는 빈 예측이면 top-k 후보를 fallback으로 사용할 수 있습니다.
        fallback_used = False
        if len(selected) == 0 and fallback_top_k > 0:
            selected = [OPERATION_LABELS[j] for j in np.argsort(-prob)[:fallback_top_k]]
            fallback_used = True

        results.append({
            "text": text,
            "signals": selected,
            "signals_ko": [LABEL_KO[x] for x in selected],
            "top_labels": top_labels,
            "fallback_used": fallback_used,
        })
    return results


def analyze_review_by_clauses(text):
    clauses = split_review_clauses(text)
    clause_results = predict_multilabel_texts(clauses, mode="labelwise", top_k=3, fallback_top_k=1)
    merged = sorted(set([sig for r in clause_results for sig in r["signals"]]))
    return {
        "original_text": text,
        "clauses": clause_results,
        "merged_signals": merged,
        "merged_signals_ko": [LABEL_KO[x] for x in merged],
    }

sample_reviews = [
    "커피는 맛있는데 직원 응대가 조금 아쉬웠어요.",
    "가격은 조금 비싸지만 디저트가 맛있어서 또 오고 싶어요.",
    "주말에는 사람이 너무 많아서 주문까지 오래 기다렸지만 직원은 친절했어요.",
    "분위기는 좋은데 테이블 정리가 늦어서 지저분하게 느껴졌어요.",
]

sample_results = [analyze_review_by_clauses(x) for x in sample_reviews]
print(json.dumps(sample_results, ensure_ascii=False, indent=2))
with open(OUTPUT_DIR / "sample_clause_predictions.json", "w", encoding="utf-8") as f:
    json.dump(sample_results, f, ensure_ascii=False, indent=2)

## 18. Store DNA 점수화

예측된 운영 신호를 매장 DNA 점수로 변환합니다. 실제 서비스에서는 POS 데이터와 결합해 점수를 보정해야 합니다.

In [ ]:
DNA_MAPPING = {
    "product_power": {"name": "제품 경쟁력", "positive": ["product_strength"], "negative": ["product_risk"]},
    "service_stability": {"name": "서비스 안정성", "positive": ["service_strength"], "negative": ["service_risk"]},
    "price_acceptance": {"name": "가격 수용성", "positive": ["price_value_strength"], "negative": ["price_resistance"]},
    "space_attractiveness": {"name": "공간 매력도", "positive": ["space_strength", "cleanliness_strength"], "negative": ["space_risk", "cleanliness_risk"]},
    "waiting_stability": {"name": "대기/혼잡 안정성", "positive": ["operation_efficiency"], "negative": ["waiting_risk"]},
    "revisit_power": {"name": "재방문 신호", "positive": ["revisit_intent"], "negative": ["revisit_risk"]},
}


def compute_store_dna_from_signals(signal_lists):
    counts = Counter()
    for signals in signal_lists:
        counts.update(signals)
    total = max(sum(counts.values()), 1)

    rows = []
    for key, info in DNA_MAPPING.items():
        pos = sum(counts[x] for x in info["positive"])
        neg = sum(counts[x] for x in info["negative"])
        raw = 50 + 50 * ((pos - neg) / max(pos + neg, 1))
        # 신호량이 너무 적으면 50점에 가까워지도록 보수적으로 조정
        volume = min((pos + neg) / max(total * 0.3, 1), 1)
        score = 50 + (raw - 50) * volume
        rows.append({
            "항목": info["name"],
            "점수": round(float(score), 1),
            "긍정신호수": int(pos),
            "부정신호수": int(neg),
        })
    return pd.DataFrame(rows).sort_values("점수", ascending=False).reset_index(drop=True)

# 샘플 예측 결과 기반 DNA 점수
sample_signal_lists = [r["merged_signals"] for r in sample_results]
dna_df = compute_store_dna_from_signals(sample_signal_lists)
dna_df.to_csv(OUTPUT_DIR / "sample_store_dna_scores.csv", index=False, encoding="utf-8-sig")
dna_df

## 19. 사장님용 해석 문장 예시

모델 출력은 최종 리포트가 아니라, LLM 또는 규칙 기반 리포트 생성기의 입력입니다.

In [ ]:
def generate_simple_store_comment(dna_df):
    strongest = dna_df.sort_values("점수", ascending=False).head(2)
    weakest = dna_df.sort_values("점수").head(2)
    strong_text = ", ".join([f"{row['항목']}({int(row['점수'])}점)" for _, row in strongest.iterrows()])
    weak_text = ", ".join([f"{row['항목']}({int(row['점수'])}점)" for _, row in weakest.iterrows()])
    return (
        f"이번 리뷰 신호를 보면 강점은 {strong_text}입니다. "
        f"반대로 우선 점검해야 할 항목은 {weak_text}입니다. "
        "다음 단계에서는 이 신호를 POS의 시간대별 매출, 메뉴별 매출, 피크타임 주문량과 결합해 실제 운영 병목을 확인해야 합니다."
    )

comment = generate_simple_store_comment(dna_df)
print(comment)
with open(OUTPUT_DIR / "sample_store_comment.txt", "w", encoding="utf-8") as f:
    f.write(comment)

## 20. 최종 모델/tokenizer 저장 및 산출물 검증

이 셀을 실행하면 서비스 추론에 필요한 HuggingFace 모델 폴더가 생성됩니다.

생성 목표 파일:

```text
config.json
model.safetensors 또는 pytorch_model.bin
tokenizer.json
tokenizer_config.json
special_tokens_map.json
metadata.json
label2id.json
id2label.json
operation_labels.json
label_thresholds.json
```

> 중요: 이 셀은 학습이 끝난 뒤 실행해야 합니다. 즉, `model`, `tokenizer`, `best_threshold`, `label_thresholds`가 준비된 상태에서 실행합니다.


In [ ]:
# ============================================================
# 20. 최종 모델/tokenizer 저장 및 산출물 검증
# ============================================================
# 목적:
# - 학습 완료된 Transformer 모델을 서비스에서 다시 불러올 수 있는 형태로 저장합니다.
# - config.json, model.safetensors 또는 pytorch_model.bin, tokenizer.json,
#   tokenizer_config.json, special_tokens_map.json 생성 여부를 검증합니다.

from pathlib import Path
import json
import shutil

# 1) 필수 객체 확인
required_runtime_objects = ["model", "tokenizer", "OPERATION_LABELS", "LABEL_KO"]
missing_runtime_objects = [name for name in required_runtime_objects if name not in globals()]
if missing_runtime_objects:
    raise NameError(
        "모델 저장에 필요한 객체가 없습니다: " + ", ".join(missing_runtime_objects) + "\n"
        "앞의 학습 셀을 먼저 순서대로 실행하세요."
    )

# 2) 저장 경로 고정
# 기존 MODEL_DIR이 있으면 사용하고, 없으면 기본 경로를 생성합니다.
BASE_DIR = globals().get("BASE_DIR", Path.cwd())
MODEL_DIR = globals().get(
    "MODEL_DIR",
    BASE_DIR / "models" / "cafe_review_signal_multilabel_v1_2"
)
MODEL_DIR = Path(MODEL_DIR)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 완전한 최종본을 위해 이전 저장물을 지우고 다시 저장합니다.
# 같은 폴더를 계속 쓰면 오래된 파일이 남아 혼동될 수 있습니다.
if MODEL_DIR.exists():
    shutil.rmtree(MODEL_DIR)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 3) 라벨 mapping을 config에도 명시적으로 반영
label2id = {label: i for i, label in enumerate(OPERATION_LABELS)}
id2label = {i: label for i, label in enumerate(OPERATION_LABELS)}

model.config.label2id = label2id
model.config.id2label = id2label
model.config.problem_type = "multi_label_classification"
model.config.num_labels = len(OPERATION_LABELS)

# 4) threshold 값 안전 처리
# best_threshold / label_thresholds가 없으면 기본 threshold로 대체합니다.
DEFAULT_THRESHOLD = globals().get("DEFAULT_THRESHOLD", 0.5)
best_threshold_to_save = globals().get("best_threshold", DEFAULT_THRESHOLD)

if "label_thresholds" in globals() and isinstance(label_thresholds, dict):
    label_thresholds_to_save = {label: float(label_thresholds.get(label, best_threshold_to_save)) for label in OPERATION_LABELS}
else:
    label_thresholds_to_save = {label: float(best_threshold_to_save) for label in OPERATION_LABELS}

MAX_LENGTH = globals().get("MAX_LENGTH", 128)
MODEL_NAME = globals().get("MODEL_NAME", "unknown")

# 5) 모델 가중치 저장
# safe_serialization=True이면 model.safetensors가 생성됩니다.
# 환경에 따라 safetensors 저장이 실패하면 pytorch_model.bin으로 자동 대체합니다.
try:
    model.save_pretrained(str(MODEL_DIR), safe_serialization=True)
    weight_save_type = "model.safetensors"
except Exception as e:
    print("safe_serialization 저장 실패. pytorch_model.bin 방식으로 재시도합니다.")
    print("원인:", repr(e))
    model.save_pretrained(str(MODEL_DIR), safe_serialization=False)
    weight_save_type = "pytorch_model.bin"

# 6) tokenizer 저장
# Fast tokenizer인 경우 tokenizer.json이 함께 생성됩니다.
tokenizer.save_pretrained(str(MODEL_DIR))

# tokenizer.json이 생성되지 않은 경우, fast tokenizer backend가 있으면 직접 저장을 시도합니다.
tokenizer_json_path = MODEL_DIR / "tokenizer.json"
if not tokenizer_json_path.exists() and getattr(tokenizer, "is_fast", False):
    try:
        tokenizer.backend_tokenizer.save(str(tokenizer_json_path))
        print("tokenizer.json 직접 저장 완료")
    except Exception as e:
        print("tokenizer.json 직접 저장 실패:", repr(e))

# 7) 서비스 추론용 메타데이터 저장
metadata = {
    "model_name": MODEL_NAME,
    "task": "cafe_review_signal_multilabel_classification_v1_2",
    "model_dir": str(MODEL_DIR),
    "labels": list(OPERATION_LABELS),
    "label_ko": LABEL_KO,
    "label2id": label2id,
    "id2label": {str(k): v for k, v in id2label.items()},
    "best_global_threshold": float(best_threshold_to_save),
    "label_wise_thresholds": label_thresholds_to_save,
    "default_threshold": float(DEFAULT_THRESHOLD),
    "max_length": int(MAX_LENGTH),
    "weight_file_type": weight_save_type,
    "notes": (
        "v1.2 multilabel cafe review signal model. "
        "Use metadata.json and label_thresholds.json for service inference."
    ),
}

with open(MODEL_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

with open(MODEL_DIR / "label2id.json", "w", encoding="utf-8") as f:
    json.dump(label2id, f, ensure_ascii=False, indent=2)

with open(MODEL_DIR / "id2label.json", "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in id2label.items()}, f, ensure_ascii=False, indent=2)

with open(MODEL_DIR / "operation_labels.json", "w", encoding="utf-8") as f:
    json.dump(list(OPERATION_LABELS), f, ensure_ascii=False, indent=2)

with open(MODEL_DIR / "label_thresholds.json", "w", encoding="utf-8") as f:
    json.dump(label_thresholds_to_save, f, ensure_ascii=False, indent=2)

# 8) 필수 파일 검증
files = sorted([p.name for p in MODEL_DIR.iterdir() if p.is_file()])

required_exact_files = [
    "config.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "metadata.json",
    "label2id.json",
    "id2label.json",
    "operation_labels.json",
    "label_thresholds.json",
]

missing_exact_files = [name for name in required_exact_files if not (MODEL_DIR / name).exists()]

has_weight_file = (MODEL_DIR / "model.safetensors").exists() or (MODEL_DIR / "pytorch_model.bin").exists()
has_tokenizer_json = (MODEL_DIR / "tokenizer.json").exists()

print("모델 저장 경로:", MODEL_DIR)
print("가중치 저장 방식:", weight_save_type)
print("저장된 파일 목록:")
for name in files:
    print(" -", name)

if missing_exact_files:
    print("\n누락된 필수 파일:", missing_exact_files)

if not has_weight_file:
    raise FileNotFoundError("model.safetensors 또는 pytorch_model.bin 파일이 생성되지 않았습니다.")

if not has_tokenizer_json:
    raise FileNotFoundError(
        "tokenizer.json 파일이 생성되지 않았습니다. "
        "AutoTokenizer가 fast tokenizer로 로드되었는지 확인하세요."
    )

if missing_exact_files:
    raise FileNotFoundError("일부 필수 파일이 누락되었습니다: " + ", ".join(missing_exact_files))

print("\n저장 검증 완료")
print("서비스 추론에 필요한 모델/tokenizer 파일이 준비되었습니다.")


## 20-1. 저장된 모델 재로드 테스트

저장된 폴더가 실제 서비스 코드에서 다시 로드되는지 확인합니다. 이 셀이 성공하면 `AutoTokenizer.from_pretrained(MODEL_DIR)`와 `AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)` 방식으로 서비스에서 사용할 수 있습니다.


In [ ]:
# ============================================================
# 20-1. 저장된 모델 재로드 테스트
# ============================================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification

reloaded_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
reloaded_model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))

print("재로드 성공")
print("모델 라벨 수:", reloaded_model.config.num_labels)
print("tokenizer fast 여부:", getattr(reloaded_tokenizer, "is_fast", None))
print("config problem_type:", reloaded_model.config.problem_type)
print("예시 id2label:", reloaded_model.config.id2label)


## 21. FastAPI 연결용 predictor 초안 저장

서비스에서는 `predict_review_signals()` 함수를 API 안에서 호출하면 됩니다.

In [ ]:
SERVICE_PREDICTOR_CODE = r'''
from pathlib import Path
import re
import json
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def split_review_clauses(text):
    text = str(text).strip()
    patterns = [r"는데\s*", r"지만\s*", r"다만\s*", r"하지만\s*", r"그런데\s*", r"그리고\s*", r"또\s*", r"반면\s*", r",", r"\."]
    parts = re.split("|".join(patterns), text)
    parts = [p.strip() for p in parts if len(p.strip()) >= 3]
    return parts if parts else [text]

class CafeReviewSignalPredictor:
    def __init__(self, model_dir):
        self.model_dir = Path(model_dir)
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(self.model_dir)
        self.model.eval()
        with open(self.model_dir / "metadata.json", "r", encoding="utf-8") as f:
            self.meta = json.load(f)
        self.labels = self.meta["labels"]
        self.label_ko = self.meta["label_ko"]
        self.label_thresholds = self.meta["label_wise_thresholds"]
        self.max_length = self.meta.get("max_length", 128)

    def predict_texts(self, texts, top_k=3, fallback_top_k=1):
        if isinstance(texts, str):
            texts = [texts]
        enc = self.tokenizer(texts, truncation=True, max_length=self.max_length, padding=True, return_tensors="pt")
        with torch.no_grad():
            logits = self.model(**enc).logits.detach().cpu().numpy()
        probs = sigmoid(logits)
        results = []
        for text, prob in zip(texts, probs):
            selected = [label for j, label in enumerate(self.labels) if prob[j] >= self.label_thresholds[label]]
            fallback_used = False
            if not selected and fallback_top_k > 0:
                selected = [self.labels[j] for j in np.argsort(-prob)[:fallback_top_k]]
                fallback_used = True
            top_idx = np.argsort(-prob)[:top_k]
            results.append({
                "text": text,
                "signals": selected,
                "signals_ko": [self.label_ko[x] for x in selected],
                "top_labels": [{"label": self.labels[j], "label_ko": self.label_ko[self.labels[j]], "score": float(prob[j])} for j in top_idx],
                "fallback_used": fallback_used,
            })
        return results

    def predict_review_signals(self, text):
        clauses = split_review_clauses(text)
        clause_results = self.predict_texts(clauses)
        merged = sorted(set(sig for r in clause_results for sig in r["signals"]))
        return {
            "original_text": text,
            "clauses": clause_results,
            "merged_signals": merged,
            "merged_signals_ko": [self.label_ko[x] for x in merged],
        }
'''

predictor_path = SERVICE_DIR / "review_signal_predictor_v1_2.py"
with open(predictor_path, "w", encoding="utf-8") as f:
    f.write(SERVICE_PREDICTOR_CODE)
print("저장 완료:", predictor_path)

## 22. 이번 v1.2에서 확인해야 할 결론

v1.2의 성공 기준은 다음입니다.

1. `empty_prediction_rate`가 v1.1보다 낮아졌는가?
2. threshold 튜닝 후 `micro_f1`, `macro_f1`이 0에서 벗어났는가?
3. top-2 또는 top-3 후보 안에 정답 라벨이 들어가는가?
4. 복합 문장에서 2개 이상의 운영 신호가 추출되는가?
5. Store DNA 점수화가 서비스 해석으로 연결되는가?

이 기준을 통과하면 다음 단계는 실제 카페 리뷰 데이터 1,000개 이상 수집과 사람 검수 라벨링입니다.

## 23. 다음 단계

1. 실제 카페 리뷰 1,000개 이상 수집
2. 사람이 검수한 다중 라벨 데이터셋 구축
3. 라벨별 최소 100개 이상 확보
4. 복합 라벨 샘플 비율을 40% 이상으로 구성
5. threshold/label-wise threshold를 실제 validation set 기준으로 재탐색
6. POS 데이터와 결합해 Store DNA 점수 보정
7. FastAPI 예측 API 연결
8. 장사비서 리포트 생성 엔진에 통합

> 핵심: v1.2는 서비스 모델 완성이 아니라, “다중 라벨 예측이 서비스형 구조로 작동하는지”를 확인하는 개선 실험입니다.

## 24. 최종 사용 방법

1. 이 노트북을 위에서 아래로 순서대로 실행합니다.
2. 학습 완료 후 `20. 최종 모델/tokenizer 저장 및 산출물 검증` 셀을 실행합니다.
3. 아래 폴더가 생성되었는지 확인합니다.

```text
models/cafe_review_signal_multilabel_v1_2/
```

4. 이 폴더 안에 다음 파일이 있어야 합니다.

```text
config.json
model.safetensors 또는 pytorch_model.bin
tokenizer.json
tokenizer_config.json
special_tokens_map.json
metadata.json
label2id.json
id2label.json
operation_labels.json
label_thresholds.json
```

5. 이 폴더를 FastAPI 또는 추론 파이프라인에서 그대로 불러오면 됩니다.



# 25. 저장된 모델로 실제 카페 리뷰 1,000개 분석하기

이 섹션은 서비스 추론 파이프라인입니다.

앞쪽 학습 과정에서 저장한 모델을 다시 로드한 뒤,  
`all_cafe_reviews_1000.csv`를 분석하여 다음 파일을 생성합니다.

```text
outputs/real_reviews_model_predictions.csv
outputs/real_reviews_signal_summary.csv
outputs/real_reviews_store_dna_report.json
outputs/real_reviews_owner_report.txt
```

이 부분은 나중에 FastAPI 서비스의 `/analyze-reviews` API로 분리할 수 있습니다.


In [ ]:

# ============================================================
# 25-1. 저장 모델 + 실제 리뷰 CSV 자동 탐색
# ============================================================

from pathlib import Path
import json, re, os, math
from collections import Counter

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 모델 폴더 후보
MODEL_DIR_CANDIDATES = [
    PROJECT_ROOT / "models" / "cafe_review_signal_multilabel_v1_2",
    PROJECT_ROOT / "models" / "cafe_review_multilabel_v1_2",
    PROJECT_ROOT / "cafe_review_signal_multilabel_v1_2",
    Path("/workspace/models/cafe_review_signal_multilabel_v1_2"),
    Path("/workspace/models/cafe_review_multilabel_v1_2"),
    Path("/mnt/data/models/cafe_review_signal_multilabel_v1_2"),
]

MODEL_DIR = None
for p in MODEL_DIR_CANDIDATES:
    if p.exists() and (p / "config.json").exists():
        MODEL_DIR = p
        break

if MODEL_DIR is None:
    raise FileNotFoundError(
        "저장된 모델 폴더를 찾지 못했습니다. 먼저 모델 저장 셀을 실행하세요.\n"
        + "\n".join([str(p) for p in MODEL_DIR_CANDIDATES])
    )

# CSV 후보
CSV_CANDIDATES = [
    PROJECT_ROOT / "all_cafe_reviews_1000.csv",
    PROJECT_ROOT / "data" / "all_cafe_reviews_1000.csv",
    Path("/workspace/all_cafe_reviews_1000.csv"),
    Path("/mnt/data/all_cafe_reviews_1000.csv"),
    Path("/mnt/data/all_cafe_reviews_1000(1).csv"),
]

CSV_PATH = None
for p in CSV_CANDIDATES:
    if p.exists():
        CSV_PATH = p
        break

if CSV_PATH is None:
    raise FileNotFoundError(
        "all_cafe_reviews_1000.csv 파일을 찾지 못했습니다.\n"
        + "\n".join([str(p) for p in CSV_CANDIDATES])
    )

print("MODEL_DIR:", MODEL_DIR)
print("CSV_PATH:", CSV_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:

# ============================================================
# 25-2. 저장된 모델/tokenizer 로드
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 저장된 operation_labels / threshold 로드
labels_path = MODEL_DIR / "operation_labels.json"
thresholds_path = MODEL_DIR / "label_thresholds.json"

if labels_path.exists():
    with open(labels_path, "r", encoding="utf-8") as f:
        OPERATION_LABELS = json.load(f)
else:
    OPERATION_LABELS = [
        "product_strength", "product_risk",
        "service_strength", "service_risk",
        "price_value_strength", "price_resistance",
        "waiting_risk",
        "space_strength", "space_risk",
        "cleanliness_strength", "cleanliness_risk",
        "revisit_intent", "revisit_risk",
        "operation_efficiency",
    ]

if thresholds_path.exists():
    with open(thresholds_path, "r", encoding="utf-8") as f:
        LABEL_THRESHOLDS = json.load(f)
else:
    LABEL_THRESHOLDS = {label: 0.5 for label in OPERATION_LABELS}

LABEL_KO = {
    "product_strength": "제품/메뉴 강점",
    "product_risk": "제품/메뉴 리스크",
    "service_strength": "서비스 강점",
    "service_risk": "서비스 리스크",
    "price_value_strength": "가격 대비 만족",
    "price_resistance": "가격 저항",
    "waiting_risk": "대기/혼잡 리스크",
    "space_strength": "공간/분위기 강점",
    "space_risk": "공간/좌석 리스크",
    "cleanliness_strength": "청결 강점",
    "cleanliness_risk": "청결 리스크",
    "revisit_intent": "재방문 의도",
    "revisit_risk": "재방문 저하",
    "operation_efficiency": "운영 효율",
}

print("모델 로드 완료")
print("device:", device)
print("라벨 수:", len(OPERATION_LABELS))
print("가중치 파일 존재:", (MODEL_DIR / "model.safetensors").exists() or (MODEL_DIR / "pytorch_model.bin").exists())


In [ ]:

# ============================================================
# 25-3. 실제 리뷰 CSV 로드 및 review_text 생성
# ============================================================

def clean_text(text):
    text = "" if pd.isna(text) else str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df_real = pd.read_csv(CSV_PATH)

print("원본 크기:", df_real.shape)
print("컬럼:", df_real.columns.tolist())

# title + snippet이 있으면 우선 사용
if "title" in df_real.columns:
    df_real["title_clean"] = df_real["title"].apply(clean_text)
else:
    df_real["title_clean"] = ""

if "snippet" in df_real.columns:
    df_real["snippet_clean"] = df_real["snippet"].apply(clean_text)
else:
    df_real["snippet_clean"] = ""

fallback_text = df_real.select_dtypes(include=["object"]).fillna("").astype(str).agg(" ".join, axis=1)
df_real["review_text"] = (df_real["title_clean"] + " " + df_real["snippet_clean"]).str.strip()

empty_mask = df_real["review_text"].str.len() == 0
df_real.loc[empty_mask, "review_text"] = fallback_text[empty_mask].apply(clean_text)

before = len(df_real)
df_real = df_real[df_real["review_text"].str.len() > 0].copy()
df_real = df_real.drop_duplicates(subset=["review_text"]).reset_index(drop=True)
after = len(df_real)

print("전처리 전:", before)
print("전처리 후:", after)
print("제거:", before - after)

display(df_real.head())


In [ ]:

# ============================================================
# 25-4. 실제 리뷰 1,000개 모델 예측
# ============================================================

def sigmoid_np(x):
    return 1 / (1 + np.exp(-x))

def predict_texts(texts, batch_size=32, max_length=160, fallback_top_k=2):
    all_probs = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        inputs = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = model(**inputs).logits.detach().cpu().numpy()

        probs = sigmoid_np(logits)
        all_probs.append(probs)

    all_probs = np.vstack(all_probs)
    pred_bin = np.zeros_like(all_probs, dtype=int)

    for j, label in enumerate(OPERATION_LABELS):
        threshold = float(LABEL_THRESHOLDS.get(label, 0.5))
        pred_bin[:, j] = (all_probs[:, j] >= threshold).astype(int)

    # 빈 예측 fallback: top-k 라벨을 후보로 부여
    for i in range(len(pred_bin)):
        if pred_bin[i].sum() == 0:
            top_idx = np.argsort(-all_probs[i])[:fallback_top_k]
            pred_bin[i, top_idx] = 1

    results = []
    for i in range(len(texts)):
        labels = [OPERATION_LABELS[j] for j, v in enumerate(pred_bin[i]) if v == 1]
        labels_ko = [LABEL_KO.get(label, label) for label in labels]
        top3_idx = np.argsort(-all_probs[i])[:3]
        top3 = [
            {
                "label": OPERATION_LABELS[j],
                "label_ko": LABEL_KO.get(OPERATION_LABELS[j], OPERATION_LABELS[j]),
                "score": float(all_probs[i, j]),
            }
            for j in top3_idx
        ]
        max_conf = float(all_probs[i].max())
        results.append({
            "predicted_signals": labels,
            "predicted_signals_ko": labels_ko,
            "top3_signals": top3,
            "max_confidence": max_conf,
            "num_predicted_labels": len(labels),
            "needs_human_review": bool(
                max_conf < 0.55
                or len(labels) >= 4
                or any(x in labels for x in ["price_resistance", "service_risk", "cleanliness_risk"])
            ),
        })

    return results

texts = df_real["review_text"].astype(str).tolist()
pred_results = predict_texts(texts, batch_size=32)

df_pred = df_real.copy()
df_pred["predicted_signals"] = [r["predicted_signals"] for r in pred_results]
df_pred["predicted_signals_ko"] = [r["predicted_signals_ko"] for r in pred_results]
df_pred["top3_signals"] = [json.dumps(r["top3_signals"], ensure_ascii=False) for r in pred_results]
df_pred["max_confidence"] = [r["max_confidence"] for r in pred_results]
df_pred["num_predicted_labels"] = [r["num_predicted_labels"] for r in pred_results]
df_pred["needs_human_review"] = [r["needs_human_review"] for r in pred_results]

pred_path = OUTPUT_DIR / "real_reviews_model_predictions.csv"
df_pred.to_csv(pred_path, index=False, encoding="utf-8-sig")

print("예측 결과 저장:", pred_path)
print("분석 리뷰 수:", len(df_pred))
display(df_pred[["review_text", "predicted_signals_ko", "max_confidence", "needs_human_review"]].head(20))


In [ ]:

# ============================================================
# 25-5. 운영 신호 요약 + Store DNA 리포트 생성
# ============================================================

signal_counter = Counter()
for labels in df_pred["predicted_signals"]:
    if isinstance(labels, list):
        signal_counter.update(labels)

signal_summary_df = pd.DataFrame([
    {
        "label": label,
        "label_ko": LABEL_KO.get(label, label),
        "count": signal_counter.get(label, 0),
        "ratio": signal_counter.get(label, 0) / max(len(df_pred), 1),
    }
    for label in OPERATION_LABELS
]).sort_values("count", ascending=False)

signal_summary_path = OUTPUT_DIR / "real_reviews_signal_summary.csv"
signal_summary_df.to_csv(signal_summary_path, index=False, encoding="utf-8-sig")

display(signal_summary_df)

STORE_DNA_RULES = {
    "제품 경쟁력": {"positive": ["product_strength"], "negative": ["product_risk"]},
    "서비스 안정성": {"positive": ["service_strength"], "negative": ["service_risk"]},
    "가격 수용성": {"positive": ["price_value_strength"], "negative": ["price_resistance"]},
    "대기/운영 안정성": {"positive": ["operation_efficiency"], "negative": ["waiting_risk"]},
    "공간 매력도": {"positive": ["space_strength"], "negative": ["space_risk"]},
    "청결 안정성": {"positive": ["cleanliness_strength"], "negative": ["cleanliness_risk"]},
    "재방문 신호": {"positive": ["revisit_intent"], "negative": ["revisit_risk"]},
}

def pick_evidence_reviews(df, labels, max_items=3):
    rows = []
    for _, row in df.iterrows():
        pred_labels = row["predicted_signals"]
        if isinstance(pred_labels, list) and any(label in pred_labels for label in labels):
            rows.append(row["review_text"])
        if len(rows) >= max_items:
            break
    return rows

def confidence_from_count(related_count, total):
    ratio = related_count / max(total, 1)
    if related_count >= 50 and ratio >= 0.05:
        return "high"
    if related_count >= 15:
        return "medium"
    return "low"

def build_store_dna_report(df):
    total = max(len(df), 1)
    report = []

    for dimension, rule in STORE_DNA_RULES.items():
        pos_labels = rule["positive"]
        neg_labels = rule["negative"]

        pos_count = sum(signal_counter.get(label, 0) for label in pos_labels)
        neg_count = sum(signal_counter.get(label, 0) for label in neg_labels)
        related_count = pos_count + neg_count

        # 기본 50점에서 긍정/부정 비율 반영
        score = 50 + 50 * ((pos_count - neg_count) / total)
        score = round(max(0, min(100, score)), 1)

        if score >= 70:
            owner_message = f"{dimension}은 현재 강점으로 활용할 수 있습니다."
        elif score >= 45:
            owner_message = f"{dimension}은 보통 수준이며, 리뷰 근거를 추가 확인하는 것이 좋습니다."
        else:
            owner_message = f"{dimension}은 우선 점검이 필요한 신호가 감지됩니다."

        report.append({
            "dimension": dimension,
            "score": score,
            "confidence": confidence_from_count(related_count, total),
            "positive_count": int(pos_count),
            "negative_count": int(neg_count),
            "related_count": int(related_count),
            "evidence_reviews": pick_evidence_reviews(df, pos_labels + neg_labels, max_items=3),
            "owner_message": owner_message,
        })

    return sorted(report, key=lambda x: x["score"], reverse=True)

store_dna_report = build_store_dna_report(df_pred)

store_dna_path = OUTPUT_DIR / "real_reviews_store_dna_report.json"
with open(store_dna_path, "w", encoding="utf-8") as f:
    json.dump(store_dna_report, f, ensure_ascii=False, indent=2)

store_dna_df = pd.DataFrame(store_dna_report)
display(store_dna_df[["dimension", "score", "confidence", "positive_count", "negative_count", "owner_message"]])

print("신호 요약 저장:", signal_summary_path)
print("Store DNA 저장:", store_dna_path)


In [ ]:

# ============================================================
# 25-6. 사장님용 요약 리포트 + 내일 할 일 3개
# ============================================================

def make_owner_report(store_dna_report, signal_summary_df, total_reviews):
    sorted_dna = sorted(store_dna_report, key=lambda x: x["score"], reverse=True)
    strengths = sorted_dna[:2]
    risks = sorted_dna[-2:]

    top_signals = signal_summary_df.head(3)

    strength_txt = ", ".join([f"{x['dimension']}({x['score']}점)" for x in strengths])
    risk_txt = ", ".join([f"{x['dimension']}({x['score']}점)" for x in risks])
    top_signal_txt = ", ".join(top_signals["label_ko"].tolist())

    actions = []

    low_dims = [x["dimension"] for x in risks]
    top_labels = signal_summary_df.head(6)["label"].tolist()

    if "price_resistance" in top_labels or "가격 수용성" in low_dims:
        actions.append("가격 저항 신호가 있는 메뉴는 단품 가격 인하보다 세트 구성과 가치 설명 문구를 먼저 테스트하세요.")
    if "waiting_risk" in top_labels or "대기/운영 안정성" in low_dims:
        actions.append("대기/혼잡 신호가 보이면 피크타임 인기 메뉴 3개를 선노출하고 주문 동선을 단순화하세요.")
    if "service_risk" in top_labels or "서비스 안정성" in low_dims:
        actions.append("서비스 리스크 리뷰는 시간대별로 묶어 보고, 바쁜 시간대 응대 멘트와 역할 분담을 먼저 점검하세요.")
    if "product_strength" in top_labels:
        actions.append("제품/메뉴 강점 리뷰 문장은 대표 메뉴 홍보 문구와 매장 소개에 재활용하세요.")
    if "revisit_intent" in top_labels:
        actions.append("재방문 의도 신호가 있는 리뷰는 쿠폰/스탬프/단골 전환 메시지와 연결하세요.")

    # 최소 3개 보장
    default_actions = [
        "POS 데이터의 시간대별 매출과 리뷰 신호를 연결해 실제 병목 시간대를 확인하세요.",
        "리뷰 근거 문장 10개를 직접 읽고 모델 라벨이 맞는지 검수하세요.",
        "상위 강점 1개와 하위 리스크 1개만 골라 이번 주 개선 실험을 설계하세요.",
    ]
    for action in default_actions:
        if len(actions) >= 3:
            break
        if action not in actions:
            actions.append(action)

    report = f"""
# 카페 리뷰 운영 신호 분석 리포트

분석 리뷰 수: {total_reviews:,}건

## 핵심 요약
이번 리뷰 데이터에서 가장 많이 감지된 운영 신호는 **{top_signal_txt}**입니다.

Store DNA 기준 강점은 **{strength_txt}**로 나타났습니다.  
반대로 우선 점검해야 할 항목은 **{risk_txt}**입니다.

## 사장님께 드리는 해석
이 분석은 리뷰를 단순 긍정/부정으로 나누는 것이 아니라, 손님의 말 속에서 매장 운영에 필요한 신호를 뽑아낸 결과입니다.  
다음 단계에서는 이 리뷰 신호를 POS의 시간대별 매출, 메뉴별 매출, 객단가, 주문 집중도와 연결해야 실제 운영 병목을 더 정확하게 판단할 수 있습니다.

## 내일 할 일 3개
1. {actions[0]}
2. {actions[1]}
3. {actions[2]}
""".strip()

    return report, actions[:3]

owner_report, recommended_actions = make_owner_report(store_dna_report, signal_summary_df, len(df_pred))

owner_report_path = OUTPUT_DIR / "real_reviews_owner_report.txt"
with open(owner_report_path, "w", encoding="utf-8") as f:
    f.write(owner_report)

print(owner_report)
print("\n저장:", owner_report_path)



# 26. Codex 서비스화 연결

이 노트북에서 서비스 코드로 옮겨야 할 핵심 기능은 아래 5개입니다.

```text
1. 모델 로드
   - AutoTokenizer.from_pretrained(MODEL_DIR)
   - AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

2. CSV 업로드/전처리
   - title + snippet → review_text
   - 중복/공백 제거

3. 다중 라벨 예측
   - sigmoid
   - label_threshold 적용
   - top3 signal 생성

4. Store DNA 변환
   - product/service/price/waiting/space/cleanliness/revisit 차원 점수화
   - confidence와 근거 리뷰 추출

5. 사장님용 리포트
   - 핵심 요약
   - 강점/리스크
   - 내일 할 일 3개
```

FastAPI 기준 첫 MVP 엔드포인트는 다음 3개면 충분합니다.

```text
GET  /health
POST /analyze-reviews
GET  /download/{job_id}/{file_type}
```

이제부터 Codex와 할 일은 **새 모델을 만드는 것**이 아니라,  
이 노트북의 후반부 추론 파이프라인을 `backend/app/services/` 코드로 분리하는 것입니다.
